# Notebook 5: Motif Semantic Interpretation

This notebook is the interpretability-first surface for the project. It does not retrain, rediscover motifs, or use `q`.

Instead, it loads the cached `nb03` motif artifacts, focuses on the **joint** branch by default, and tries to answer a human question: what do these motifs actually look like?

The notebook is image-first:
- top exemplars
- borderline members
- near misses
- active-cell overlays
- approximate crops
- short plain-English interpretation


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPO_URL = "https://github.com/jacobposchl/model-interpretability.git"
REPO_DIR = Path("/content/model_interpretability")
IN_COLAB = "google.colab" in sys.modules
LOCAL_REPO_DIR = Path.cwd().resolve()
if not (LOCAL_REPO_DIR / "configs" / "flow").exists() and (LOCAL_REPO_DIR.parent / "configs" / "flow").exists():
    LOCAL_REPO_DIR = LOCAL_REPO_DIR.parent

if REPO_DIR.parent.exists() and IN_COLAB:
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    else:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_DIR)], check=True)
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_ROOT = REPO_DIR if REPO_DIR.exists() else LOCAL_REPO_DIR
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)


In [ ]:
from pathlib import Path
import json
import math
import subprocess
import sys

import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch

try:
    from flow_circuits.data import build_cifar10_splits
    from flow_circuits.evaluation import (
        CIFAR10_CLASS_NAMES,
        run_motif_borderline_member_experiment,
        run_motif_semantic_report_experiment,
        run_motif_spatial_footprint_experiment,
    )
    from flow_circuits.training import (
        collect_interpretability_outputs,
        load_components_from_checkpoint,
        load_yaml_config,
    )
except ModuleNotFoundError:
    REPO_DIR = Path("/content/model_interpretability")
    if REPO_DIR.exists() and str(REPO_DIR) not in sys.path:
        sys.path.insert(0, str(REPO_DIR))
    else:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(Path.cwd())], check=True)
    from flow_circuits.data import build_cifar10_splits
    from flow_circuits.evaluation import (
        CIFAR10_CLASS_NAMES,
        run_motif_borderline_member_experiment,
        run_motif_semantic_report_experiment,
        run_motif_spatial_footprint_experiment,
    )
    from flow_circuits.training import (
        collect_interpretability_outputs,
        load_components_from_checkpoint,
        load_yaml_config,
    )


## Parameters


In [ ]:
CONFIG_NAME = "resnet18_z_workflow"
CONFIG_PATH = Path("configs/flow/resnet18_aligned.yaml")
PRIMARY_BRANCH = "joint"
REFERENCE_BRANCH = "frozen"
TOP_MOTIFS_TO_RENDER = 8
MOTIF_IDS = "top"
MAX_IMAGES = 2000
SHOW_REFERENCE_COMPARISON = True
FORCE_RERUN = False

DRIVE_ROOT = Path("/content/drive/MyDrive/flow_circuits") if Path("/content/drive/MyDrive").exists() else Path("artifacts/flow_circuits")
NB03_OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb03_z_motif_discovery_and_analysis" / CONFIG_NAME
OUTPUT_DIR = DRIVE_ROOT / "notebook_runs" / "nb05_motif_semantic_interpretation" / CONFIG_NAME


In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
base_config = load_yaml_config(CONFIG_PATH)
loaders = build_cifar10_splits(
    data_dir=base_config["data"]["data_dir"],
    batch_size=base_config["data"]["batch_size"],
    num_workers=base_config["data"].get("num_workers", 4),
    seed=base_config["data"].get("seed", 0),
    augment_fit=base_config["data"].get("augment_fit", True),
    download=base_config["data"].get("download", True),
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
branch_artifacts = {
    "joint": NB03_OUTPUT_DIR / "joint" / "joint_clean_motifs.json",
    "frozen": NB03_OUTPUT_DIR / "frozen" / "frozen_clean_motifs.json",
}
if PRIMARY_BRANCH not in branch_artifacts:
    raise ValueError(f"Unknown PRIMARY_BRANCH: {PRIMARY_BRANCH}")
if not branch_artifacts[PRIMARY_BRANCH].exists():
    raise FileNotFoundError(
        f"Primary motif artifact is missing: {branch_artifacts[PRIMARY_BRANCH]}. "
        "Run nb03 first, or point PRIMARY_BRANCH at a branch with cached motifs."
    )

print("=" * 72)
print("Notebook 5 Workflow")
print("=" * 72)
print("What this notebook is doing:")
print("  1. Load cached motif artifacts from nb03. No motif rediscovery happens here.")
print("  2. Collect a manageable set of discovery images for the chosen branch.")
print("  3. Turn each motif into a semantic card with exemplars, borderline cases, and near misses.")
print("  4. Build simple spatial overlays and crop specs so we can inspect where the motif lives.")
print()
print("How to read the outputs:")
print("  motif size: number of images inside the retained motif family.")
print("  mean cluster stability: how consistently the source node-clusters survived discovery.")
print("  supporting layers / depth span: how multi-layer the motif is.")
print("  class purity: how concentrated the motif is on one class; useful but not a proof of mechanism.")
print("  heuristic tags: cautious descriptive hints, not strong claims.")
print()
print(f"Primary branch      : {PRIMARY_BRANCH}")
print(f"Reference branch    : {REFERENCE_BRANCH}")
print(f"Primary artifact    : {branch_artifacts[PRIMARY_BRANCH]}")
print(f"Reference artifact  : {branch_artifacts.get(REFERENCE_BRANCH)}")
print(f"Discovery max imgs  : {MAX_IMAGES}")
print(f"Top motifs to show  : {TOP_MOTIFS_TO_RENDER}")
print(f"Device              : {device}")
print(f"Output directory    : {OUTPUT_DIR}")

_PROGRESS_STATE = {}

def _progress_logger(**event):
    checkpoint_tag = event.get("checkpoint_tag", "?")
    experiment = event.get("experiment", "?")
    stage = event.get("stage", "?")
    completed = event.get("completed")
    total = event.get("total")
    elapsed = event.get("elapsed_seconds")
    eta = event.get("eta_seconds")
    message = event.get("message", "")
    key = (checkpoint_tag, experiment, stage)
    should_print = False
    bucket = None
    if completed is None or total is None:
        should_print = True
    else:
        total = max(int(total), 1)
        completed = int(completed)
        if completed <= 1 or completed >= total:
            should_print = True
        elif total <= 10:
            should_print = True
        else:
            bucket = int((100 * completed) / total) // 25
        if bucket is not None and _PROGRESS_STATE.get(key) != bucket:
            should_print = True
            _PROGRESS_STATE[key] = bucket
    if not should_print:
        return
    parts = [f"[{checkpoint_tag}]", experiment, stage]
    if completed is not None and total is not None:
        parts.append(f"{completed}/{total}")
    if elapsed is not None:
        parts.append(f"elapsed={elapsed:.0f}s")
    if eta is not None:
        parts.append(f"eta={eta:.0f}s")
    if message:
        parts.append(message)
    print(" | ".join(parts), flush=True)

def _load_json(path: Path):
    return json.loads(path.read_text(encoding="utf-8"))

def _run_or_load(path: Path, label: str, fn):
    if path.exists() and not FORCE_RERUN:
        print(f"  [cache hit] {label}: {path.name}", flush=True)
        return _load_json(path)
    print(f"  [compute] {label}: {path.name}", flush=True)
    result = fn(path)
    path.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(f"  [saved] {label}: {path.name}", flush=True)
    return result

def _load_branch_runtime(branch: str):
    artifact_path = branch_artifacts[branch]
    motif_artifact = _load_json(artifact_path)
    checkpoint_path = motif_artifact["metadata"]["checkpoint_path"]
    print("\n" + "-" * 72)
    print(f"Branch: {branch}")
    print(f"Checkpoint: {Path(checkpoint_path).name}")
    print(f"Artifact: {artifact_path}")
    print("Collecting display-ready discovery outputs...", flush=True)
    components = load_components_from_checkpoint(checkpoint_path, device)
    outputs = collect_interpretability_outputs(
        components,
        loaders["discovery"],
        device=device,
        max_images=MAX_IMAGES,
        progress_callback=lambda **event: _progress_logger(
            checkpoint_tag=branch,
            experiment="motif_interpretation",
            stage="data_collection",
            completed=event.get("batch_idx"),
            total=event.get("total_batches"),
            message="collecting z/images for semantic inspection",
        ),
    )
    print(f"Collected {outputs['z'].shape[0]} images for {branch}.", flush=True)
    branch_dir = OUTPUT_DIR / branch
    branch_dir.mkdir(parents=True, exist_ok=True)
    return motif_artifact, outputs, branch_dir

primary_motif_artifact, PRIMARY_OUTPUTS, primary_branch_dir = _load_branch_runtime(PRIMARY_BRANCH)
print("Artifacts:")
for artifact_name in ["semantic_report.json", "borderline_members.json", "spatial_footprints.json"]:
    artifact_path = primary_branch_dir / artifact_name
    status = "exists" if artifact_path.exists() else "missing"
    print(f"  {status:<7} {artifact_name}")

PRIMARY_SEMANTIC_REPORT = _run_or_load(
    primary_branch_dir / "semantic_report.json",
    "semantic report",
    lambda cache_path: run_motif_semantic_report_experiment(
        primary_motif_artifact,
        PRIMARY_OUTPUTS,
        checkpoint_tag=PRIMARY_BRANCH,
        motif_ids=MOTIF_IDS,
        topk=TOP_MOTIFS_TO_RENDER,
        output_path=cache_path,
        progress_callback=_progress_logger,
    ),
)
PRIMARY_BORDERLINE_REPORT = _run_or_load(
    primary_branch_dir / "borderline_members.json",
    "borderline member report",
    lambda cache_path: run_motif_borderline_member_experiment(
        primary_motif_artifact,
        PRIMARY_OUTPUTS,
        checkpoint_tag=PRIMARY_BRANCH,
        motif_ids=MOTIF_IDS,
        topk=TOP_MOTIFS_TO_RENDER,
        output_path=cache_path,
        progress_callback=_progress_logger,
    ),
)
PRIMARY_SPATIAL_REPORT = _run_or_load(
    primary_branch_dir / "spatial_footprints.json",
    "spatial footprint report",
    lambda cache_path: run_motif_spatial_footprint_experiment(
        primary_motif_artifact,
        PRIMARY_OUTPUTS,
        checkpoint_tag=PRIMARY_BRANCH,
        semantic_report=PRIMARY_SEMANTIC_REPORT,
        output_path=cache_path,
        progress_callback=_progress_logger,
    ),
)

REFERENCE_SEMANTIC_REPORT = None
if SHOW_REFERENCE_COMPARISON and REFERENCE_BRANCH in branch_artifacts and branch_artifacts[REFERENCE_BRANCH].exists():
    reference_motif_artifact, REFERENCE_OUTPUTS, reference_branch_dir = _load_branch_runtime(REFERENCE_BRANCH)
    REFERENCE_SEMANTIC_REPORT = _run_or_load(
        reference_branch_dir / "semantic_report.json",
        "reference semantic report",
        lambda cache_path: run_motif_semantic_report_experiment(
            reference_motif_artifact,
            REFERENCE_OUTPUTS,
            checkpoint_tag=REFERENCE_BRANCH,
            motif_ids=MOTIF_IDS,
            topk=min(TOP_MOTIFS_TO_RENDER, 4),
            output_path=cache_path,
            progress_callback=_progress_logger,
        ),
    )
else:
    print("\nReference comparison disabled or reference artifact missing.")


In [ ]:
EXEMPLAR_LOOKUP = {int(item["motif_id"]): item for item in PRIMARY_SEMANTIC_REPORT["exemplar_sets"]}
SPATIAL_LOOKUP = {int(item["motif_id"]): item for item in PRIMARY_SPATIAL_REPORT["overlay_specs"]}
CROP_LOOKUP = {int(item["motif_id"]): item for item in PRIMARY_SPATIAL_REPORT["crop_specs"]}


def _to_image(row_index: int, outputs: dict) -> torch.Tensor:
    return outputs["images"][int(row_index)].permute(1, 2, 0).clamp(0.0, 1.0)


def _set_title(ax, title: str):
    ax.set_title(title, fontsize=10)
    ax.axis("off")


def _label_name(label: int) -> str:
    if 0 <= int(label) < len(CIFAR10_CLASS_NAMES):
        return CIFAR10_CLASS_NAMES[int(label)]
    return str(label)


def _render_image_grid(selection: dict, outputs: dict, title: str, max_cols: int = 3):
    rows = selection.get("row_indices", [])
    labels = selection.get("labels", [])
    scores = selection.get("scores", [])
    if not rows:
        print(f"  {title}: no images", flush=True)
        return
    n_items = len(rows)
    ncols = min(max_cols, n_items)
    nrows = math.ceil(n_items / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.0 * nrows))
    axes = axes if isinstance(axes, (list, tuple)) else [axes] if not hasattr(axes, "ravel") else list(axes.ravel())
    for ax, row_index, label, score in zip(axes, rows, labels, scores):
        ax.imshow(_to_image(row_index, outputs))
        _set_title(ax, f"{_label_name(label)} | score={score:.2f}")
    for ax in axes[n_items:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


def _render_overlay_grid(overlay_spec: dict, outputs: dict, title: str, max_cols: int = 3):
    images = overlay_spec.get("images", [])
    if not images:
        print(f"  {title}: no overlay images", flush=True)
        return
    n_items = len(images)
    ncols = min(max_cols, n_items)
    nrows = math.ceil(n_items / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.0 * nrows))
    axes = axes if isinstance(axes, (list, tuple)) else [axes] if not hasattr(axes, "ravel") else list(axes.ravel())
    for ax, item in zip(axes, images):
        ax.imshow(_to_image(item["row_index"], outputs))
        for box in item.get("boxes", []):
            color = "lime" if box.get("kind") == "representative" else "orange"
            rect = patches.Rectangle(
                (box["x0"], box["y0"]),
                box["x1"] - box["x0"],
                box["y1"] - box["y0"],
                linewidth=2,
                edgecolor=color,
                facecolor="none",
            )
            ax.add_patch(rect)
        _set_title(ax, _label_name(item["label"]))
    for ax in axes[n_items:]:
        ax.axis("off")
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


def _crop_tensor(image_tensor: torch.Tensor, box: dict | None):
    if not box:
        return None
    x0, y0, x1, y1 = box["x0"], box["y0"], box["x1"], box["y1"]
    return image_tensor[y0:y1, x0:x1, :]


def _render_crop_strip(crop_spec: dict, exemplar_rows: list[int], outputs: dict, title: str):
    if not exemplar_rows:
        print(f"  {title}: no exemplar crops", flush=True)
        return
    row_index = exemplar_rows[0]
    image_tensor = _to_image(row_index, outputs)
    representative_crop = _crop_tensor(image_tensor, crop_spec.get("representative_crop"))
    union_crop = _crop_tensor(image_tensor, crop_spec.get("union_crop"))
    per_layer = crop_spec.get("per_layer_crops", [])
    ncols = 2 + len(per_layer)
    fig, axes = plt.subplots(1, ncols, figsize=(3.0 * ncols, 3.0))
    if not isinstance(axes, (list, tuple)):
        axes = list(axes.ravel())
    crops = [
        (representative_crop, "representative"),
        (union_crop, "union"),
    ] + [
        (_crop_tensor(image_tensor, item.get("union_crop")), f"layer {item['layer_idx']}") for item in per_layer
    ]
    for ax, (crop, label) in zip(axes, crops):
        if crop is None or crop.numel() == 0:
            ax.axis("off")
            continue
        ax.imshow(crop)
        _set_title(ax, label)
    fig.suptitle(title, fontsize=12)
    plt.tight_layout()
    plt.show()


for card in PRIMARY_SEMANTIC_REPORT["motif_cards"]:
    motif_id = int(card["motif_id"])
    exemplars = EXEMPLAR_LOOKUP[motif_id]
    print("=" * 72)
    print(f"Motif {motif_id} | branch={PRIMARY_BRANCH}")
    print("=" * 72)
    print(
        f"dominant_class={card['dominant_class_name']} | purity={card['class_purity']:.3f} | "
        f"stability={card['mean_cluster_stability']:.3f} | layers={card['supporting_layers']} | "
        f"depth_span={card['depth_span']} | topology={card['topology_type']}"
    )
    print(f"heuristic_tags={', '.join(card['heuristic_tags'])}")
    print(f"representative_node={card['representative_node']} | active_nodes={card['active_nodes']}")
    histogram_text = ", ".join(
        f"{item['label_name']}:{item['count']}" for item in card["class_histogram"]
    ) or "none"
    print(f"class_histogram={histogram_text}")
    _render_image_grid(exemplars["top_exemplars"], PRIMARY_OUTPUTS, f"Motif {motif_id}: top exemplars")
    _render_image_grid(exemplars["borderline_members"], PRIMARY_OUTPUTS, f"Motif {motif_id}: borderline members")
    _render_image_grid(exemplars["near_misses"], PRIMARY_OUTPUTS, f"Motif {motif_id}: near misses")


In [ ]:
for card in PRIMARY_SEMANTIC_REPORT["motif_cards"]:
    motif_id = int(card["motif_id"])
    overlay_spec = SPATIAL_LOOKUP[motif_id]
    crop_spec = CROP_LOOKUP[motif_id]
    exemplars = EXEMPLAR_LOOKUP[motif_id]
    print("-" * 72)
    print(f"Spatial footprint for motif {motif_id}")
    print("-" * 72)
    _render_overlay_grid(overlay_spec, PRIMARY_OUTPUTS, f"Motif {motif_id}: active-cell overlays")
    _render_crop_strip(crop_spec, exemplars["top_exemplars"].get("row_indices", []), PRIMARY_OUTPUTS, f"Motif {motif_id}: approximate crops")


In [ ]:
summary = PRIMARY_SEMANTIC_REPORT["summary"]
print("=" * 72)
print("Final Interpretation")
print("=" * 72)
print(f"Primary branch: {PRIMARY_BRANCH}")
print(f"n_ranked_motifs={summary['n_ranked_motifs']} | mean_purity={summary['mean_purity']:.3f} | mean_supporting_layers={summary['mean_supporting_layers']:.3f} | mean_depth_span={summary['mean_depth_span']:.3f}")
print(f"topology_counts={summary['topology_counts']}")
print(f"tag_counts={summary['tag_counts']}")
print()
print("Plain-English interpretation:")
if summary["mean_purity"] >= 0.9:
    print("- The top motifs look semantically clean overall: most are strongly concentrated on one class or visual pattern.")
elif summary["mean_purity"] >= 0.75:
    print("- The top motifs look moderately clean: there is real semantic concentration, but not every motif is sharply class-specific.")
else:
    print("- The top motifs look mixed: they are probably capturing broader or noisier visual structure rather than one clean concept.")

if summary["mean_supporting_layers"] >= 4.0:
    print("- These motifs are genuinely multi-layer. They appear to persist across depth rather than collapsing to one anchor node.")
elif summary["mean_supporting_layers"] >= 2.0:
    print("- The motifs are multi-layer, but still fairly compact. This is enough to support a flow-style interpretation.")
else:
    print("- The motifs are shallow, so interpretation should be cautious: some may still behave more like local anchors than real flow motifs.")

if summary["topology_counts"].get("depth_like", 0) >= summary["topology_counts"].get("fragmented", 0):
    print("- The branch looks interpretable overall: depth-like motifs dominate the top-ranked set.")
else:
    print("- Fragmented motifs still dominate, so the branch may need more qualitative inspection before we call it well organized.")

if summary["tag_counts"].get("background_like_candidate", 0) > summary["tag_counts"].get("object_part_candidate", 0):
    print("- Many of the top motifs may lean on border or context cues. Look closely at the overlays to separate shortcut-like patterns from real object structure.")
else:
    print("- The top motifs look more object-part-like than background-like, which is encouraging for interpretability.")

if REFERENCE_SEMANTIC_REPORT is not None:
    ref = REFERENCE_SEMANTIC_REPORT["summary"]
    print()
    print("Reference comparison:")
    print(f"- {REFERENCE_BRANCH}: mean_purity={ref['mean_purity']:.3f}, mean_supporting_layers={ref['mean_supporting_layers']:.3f}, mean_depth_span={ref['mean_depth_span']:.3f}")
    if summary["mean_purity"] > ref["mean_purity"]:
        print(f"- The {PRIMARY_BRANCH} branch looks cleaner than the {REFERENCE_BRANCH} branch on the motifs shown here.")
    elif summary["mean_purity"] < ref["mean_purity"]:
        print(f"- The {REFERENCE_BRANCH} branch looks cleaner than the {PRIMARY_BRANCH} branch on the motifs shown here.")
    else:
        print("- The two branches are about tied on purity in this semantic view.")
    if summary["mean_depth_span"] > ref["mean_depth_span"]:
        print(f"- The {PRIMARY_BRANCH} motifs also span more depth on average, which supports the flow-motif story.")
    elif summary["mean_depth_span"] < ref["mean_depth_span"]:
        print(f"- The {REFERENCE_BRANCH} motifs span more depth on average in this view.")

print()
print("What to do with these motifs next:")
print("- Use the overlays and crops to decide whether a motif looks object-part-like, texture-like, background-like, or mixed.")
print("- Pick a few especially clean motifs and follow them into intervention or corruption case studies.")
print("- If one motif family looks especially meaningful, use its motif id directly in future notebooks for focused inspection.")
